## Классификация текстов с использованием предобученных языковых моделей.

В данном задании вам предстоит обратиться к задаче классификации текстов и решить ее с использованием предобученной модели BERT.

In [126]:
import json
# do not change the code in the block below
# __________start of block__________
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

%matplotlib inline
# __________end of block__________

Обратимся к набору данных SST-2. Holdout часть данных (которая понадобится вам для посылки) доступна по ссылке ниже.

In [127]:
# do not change the code in the block below
# __________start of block__________

!wget https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
# __________end of block__________

--2024-11-17 20:16:39--  https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


HTTP request sent, awaiting response... 200 OK
Length: 51581 (50K) [text/plain]
Saving to: ‘texts_holdout.json.1’

texts_holdout.json. 100%[===================>]  50.37K  --.-KB/s    in 0.04s   

2024-11-17 20:16:39 (1.16 MB/s) - ‘texts_holdout.json.1’ saved [51581/51581]



In [128]:
# do not change the code in the block below
# __________start of block__________
df = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
)
texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]
texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]
with open("texts_holdout.json") as iofile:
    texts_holdout = json.load(iofile)
# __________end of block__________

Весь остальной код предстоит написать вам.

Для успешной сдачи на максимальный балл необходимо добиться хотя бы __84.5% accuracy на тестовой части выборки__.

### Загрузка необходимых библиотек

In [129]:
import torch

### Определение параметров среды

In [130]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('{} device is available'.format(device))

cuda device is available


### Загрузка данных

Изучим датасет:

In [131]:
df.head()

,0,1
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0
3,this is a visually stunning rumination on love...,1
4,jonathan parker 's bartleby should have been t...,1


In [132]:
df[1].value_counts()

1
1    3610
0    3310
Name: count, dtype: int64

Датасет содержит два столбца. Первый столбцец - текст, второй столбце - метка класса. Меток класса две, это бинарная классификация.
Загрузим данные:

In [141]:
from datasets import Dataset, DatasetDict, Value, Features

# Переведем данные в формат, который принимает библиотека datasets
df = df.rename(columns={0: 'text', 1: 'label'})

features = Features({
    'text': Value('string'),
    'label': Value('int8'),
})

train_dataset = Dataset.from_pandas(df[:5000], split="train", features=features)
test_dataset = Dataset.from_pandas(df[5000:], split="train", features=features)

def my_gen():
    for text in texts_holdout:
        yield {"text": text, "label": None}

holdout_dataset = Dataset.from_generator(my_gen, split="test")


dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset,
    'holdout': holdout_dataset
})

print(dataset['train'].features)
dataset

{'text': Value(dtype='string', id=None), 'label': Value(dtype='int8', id=None)}


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1920
    })
    holdout: Dataset({
        features: ['text', 'label'],
        num_rows: 500
    })
})

Посмотрим на максимальную длинну строк для каждой части датасета:

In [55]:
max_lengths = {}

for split in dataset:
    text_lengths = [len(item["text"]) for item in dataset[split] if item["text"] is not None]
    max_src_length = max(text_lengths) if text_lengths else 0
    max_lengths[split] = {"text": max_src_length}

print(max_lengths)

{'train': {'text': 271}, 'test': {'text': 262}}


Самая длинная строка во всем датасете 271 символ. Посмотрим какие символы используются в датасете:

In [56]:
# Функция для подсчета уникальных символов
def count_unique_characters(dataset, feature_name):
    unique_characters = set()
    for split in dataset.keys():
        for example in dataset[split][feature_name]:
            if example is not None:
                unique_characters.update(example)
    return len(unique_characters), unique_characters

# Подсчет уникальных символов во всех частях датасета
num_unique_chars, unique_chars = count_unique_characters(dataset, "text")

print(f"Количество уникальных символов: {num_unique_chars}")
print(f"Уникальные символы: {unique_chars}")

Количество уникальных символов: 42
Уникальные символы: {'8', 'q', 'd', 'j', 'y', 'n', 'o', 'c', '5', '2', '!', 'p', '6', 'e', 's', '`', '1', 'g', 'i', 'w', 'v', 't', ' ', ',', "'", '9', 'x', 'l', '3', 'h', 'k', '7', 'm', '?', 'u', '4', 'b', 'z', 'f', 'a', 'r', '0'}


В датасете используется 42 символа, при этом нет заглавных букв.

### Подготовка данных

Выберем модель Bert без учёта регистра:

In [57]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


Определим небольшую функцию для токенизации текста в датасете и токенизируем его (не залаем `max_length`, в этом случае используется параметр выбранной модели):

In [58]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

Используем DataCollatorWithPadding, чтобы создать батч примеров:

In [59]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Обучение модели

Загрузим модель:

In [60]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Так как классификация бинарная и классы достаточно сбалансированны, можем использовать метрику `accuracy`:

In [79]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy_score = metric.compute(predictions=predictions, references=labels)

    return accuracy_score

Определим параметры дообучения модели:

In [ ]:
batch_size = 16
learning_rate = 2e-5
epochs = 10

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./Model/{model_checkpoint}-finetuned",     # Каталог для сохранения модели
    learning_rate=learning_rate,                            # Скорость обучения
    per_device_train_batch_size=batch_size,                 # Количество обучающих примеров на устройство
    per_device_eval_batch_size=batch_size,                  # Количество тестовых примеров на устройство
    num_train_epochs=epochs,                                # Количество эпох обучения
    weight_decay=0.01,                                      # Коэффициент L2 регуляризации (0 = отсутствует). Штрафует большие веса модели, предотвращая переобучение
    fp16=True,                                              # Использовать 16-битные вычисления
    push_to_hub=True,                                       # Публиковать модель на Hugging Face Hub
    hub_private_repo = True,                                # Использовать для хранения модели приватный репозиторий
    optim = "adamw_torch",                                  # Оптимизатор AdamW из PyTorch
    load_best_model_at_end=True,                            # Загрузка лучшей модели в конце обучения
    eval_strategy="epoch",                                  # Стратегия оценки модели - каждую эпоху
    save_strategy = "epoch",                                # Стратегия сохранения модели - каждую эпоху
    metric_for_best_model="accuracy",                       # Метрика для сохранения лучшей модели
    greater_is_better = True,                               # Лучшая модель - с наибольшим значением метрики
    save_total_limit=3,                                     # Максимальное количество сохраненных контрольных точек модели в процессе обучения
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipykernel_799/2997506900.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.802925,0.879167
2,0.055600,0.627227,0.888542
3,0.055600,0.769706,0.884896
4,0.037200,0.802338,0.883333
5,0.010300,0.795999,0.885417


<transformers.trainer_utils.EvalPrediction object at 0x7f0a0e882270> <class 'transformers.trainer_utils.EvalPrediction'>
Current accuracy: {'accuracy': 0.8791666666666667}, predictions: [0 1 1 ... 0 0 1], labels: [0 1 1 ... 0 0 1]
<transformers.trainer_utils.EvalPrediction object at 0x7f08e65b1eb0> <class 'transformers.trainer_utils.EvalPrediction'>
Current accuracy: {'accuracy': 0.8885416666666667}, predictions: [0 1 1 ... 0 0 1], labels: [0 1 1 ... 0 0 1]
<transformers.trainer_utils.EvalPrediction object at 0x7f091af42660> <class 'transformers.trainer_utils.EvalPrediction'>
Current accuracy: {'accuracy': 0.8848958333333333}, predictions: [0 1 1 ... 0 0 1], labels: [0 1 1 ... 0 0 1]
<transformers.trainer_utils.EvalPrediction object at 0x7f091ad14860> <class 'transformers.trainer_utils.EvalPrediction'>
Current accuracy: {'accuracy': 0.8833333333333333}, predictions: [0 1 1 ... 0 0 1], labels: [0 1 1 ... 0 0 1]
<transformers.trainer_utils.EvalPrediction object at 0x7f08ec1703b0> <class 

TrainOutput(global_step=1565, training_loss=0.03333147177680994, metrics={'train_runtime': 66.0862, 'train_samples_per_second': 378.294, 'train_steps_per_second': 23.681, 'total_flos': 286059207791424.0, 'train_loss': 0.03333147177680994, 'epoch': 5.0})

Отправляем модель в Hub:

In [ ]:
trainer.push_to_hub()

### Формируем вероятности принадлежности первому положительному классу для всего датасета

Загрузим наилучшую модель полученную в результате обучения с Hugging Face Hub:

In [ ]:
model_checkpoint = "artyomboyko/distilbert-base-uncased-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint)
model.to(device)

Получим принадлежность к первому классу для каждого примера:

In [ ]:
def determine_probability(example):
    inputs = tokenizer(example["text"], return_tensors="pt").to(device)
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    positive_class_prob = probabilities[0][1].item()
    example["probability_1"] = positive_class_prob
    return example

# Применение функцию получения вероятности ко всему датасету
submission = dataset.map(determine_probability)

submission

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'probability_1'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label', 'probability_1'],
        num_rows: 1920
    })
    holdout: Dataset({
        features: ['text', 'label', 'probability_1'],
        num_rows: 500
    })
})

### Сдача задания в контест
Сохраните в словарь `out_dict` вероятности принадлежности к первому (положительному) классу

In [153]:
out_dict = {
    'train': [probability for probability in submission['train']['probability_1']], # list of length 5000 with probas
    'test': [probability for probability in submission['test']['probability_1']], # list of length 1920 with probas
    'holdout': [probability for probability in submission['holdout']['probability_1']] # list of length 500 with probas
}

Несколько `assert`'ов для проверки вашей посылки:

In [162]:
assert isinstance(out_dict["train"], list), "Object must be a list of floats"
assert isinstance(out_dict["train"][0], float), "Object must be a list of floats"
assert (len(out_dict["train"]) == 5000), "The predicted probas list length does not match the train set size"

assert isinstance(out_dict["test"], list), "Object must be a list of floats"
assert isinstance(out_dict["test"][0], float), "Object must be a list of floats"
assert (len(out_dict["test"]) == 1920), "The predicted probas list length does not match the test set size"

assert isinstance(out_dict["holdout"], list), "Object must be a list of floats"
assert isinstance(out_dict["holdout"][0], float), "Object must be a list of floats"
assert (len(out_dict["holdout"]) == 500), "The predicted probas list length does not match the holdout set size"

Запустите код ниже для генерации посылки.

In [163]:
# do not change the code in the block below
# __________start of block__________
FILENAME = "submission_dict_hw_text_classification_with_bert.json"

with open(FILENAME, "w") as iofile:
    json.dump(out_dict, iofile)
print(f"File saved to `{FILENAME}`")
# __________end of block__________

File saved to `submission_dict_hw_text_classification_with_bert.json`


На этом задание завершено. Поздравляем!

# Дополнительные материалы

- [Fine-tune a pretrained model](https://huggingface.co/docs/transformers/training)
- [Text classification](https://huggingface.co/docs/transformers/v4.17.0/en/tasks/sequence_classification)
- [Metric: accuracy](https://huggingface.co/spaces/evaluate-metric/accuracy)